## 1.0 Converting TEMPO files to parquet files for all hours ##

This file works to: 
- Convert all tempo netcdf files into parquet files that are easier to work with pandas and matplotlib
- Creates folders of parquet files per month, then reads folders to one combined parquet file per month

Steps:
1) Opens files lazily and creates product group file, support data file, and geolocation file to get the necessary variables
2) Creates dataset with no2, no2 uncertainty, flag, effective cloud fraction, snow ice fraction, solar zenith angle, latitude, longitude, and time
3) Makes a mask for quality filtering and filters to a smaller region
4) Writes the data into parquet file chunks that are saved to a folder with all parquet files inside
5) Makes a combined parquet file with all data per month


In [1]:
import os
import pandas as pd
import xarray as xr
from tqdm import tqdm
import dask.dataframe as dd
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from tqdm import tqdm

/Users/toireasa.marie/miniconda3/envs/plot_env/lib/python3.11/site-packages/dask/dataframe/_pyarrow_compat.py:23: UserWarning: You are using pyarrow version 13.0.0 which is known to be insecure. See https://www.cve.org/CVERecord?id=CVE-2023-47248 for further details. Please upgrade to pyarrow>=14.0.1 or install pyarrow-hotfix to patch your current version.
  warnings.warn(


In [3]:
def process_no2_files_to_combined_parquet(folder_path, output_parquet, lon_min, lon_max, lat_min, lat_max):
    temp_parquet_dir = os.path.join(folder_path, "_temp_parquets")
    os.makedirs(temp_parquet_dir, exist_ok=True)

    files =[f for f in os.listdir(folder_path) if f.endswith('.nc')]

    for file in tqdm(files, desc="Processing Files", unit="file"):
        file_path = os.path.join(folder_path, file)

        try:
            #opens lazily and ensure proper closing-- I learned this was faster, especially the chuncks partt
            with xr.open_dataset(file_path, group='product', decode_times=False, engine="netcdf4", chunks={}) as product_ds, \
                 xr.open_dataset(file_path, group='support_data', decode_times=False, engine="netcdf4", chunks={}) as support_ds, \
                 xr.open_dataset(file_path, group='geolocation', decode_times=False, engine='netcdf4', chunks={}) as geolocation_ds, \
                 xr.open_dataset(file_path, group=None, decode_times=False, engine="netcdf4", chunks={}) as root_ds:

                #variabels
                no2 = product_ds['vertical_column_troposphere']
                uncertainty = product_ds['vertical_column_troposphere_uncertainty']
                flag = product_ds['main_data_quality_flag']
                eff_cloud = support_ds['eff_cloud_fraction']
                amf_cloud = support_ds['amf_cloud_fraction']
                snow_ice = support_ds['snow_ice_fraction']
                solar_zenith = geolocation_ds['solar_zenith_angle']
                lats = root_ds['latitude']
                lons = root_ds['longitude']
                time_raw = root_ds['time']
                time  = xr.decode_cf(xr.Dataset({'time': time_raw}))['time']

                #build single dataset
                ds = xr.Dataset({
                    'no2': no2,
                    'no2_uncertainty': uncertainty,
                    'flag': flag,
                    'eff_cloud_fraction': eff_cloud,
                    'amf_cloud_fraction': amf_cloud,
                    'snow_ice_fraction': snow_ice,
                    'solar_zenith_angle':solar_zenith,
                    'latitude': lats,
                    'longitude': lons,
                    'time': time })

                #make chunking consistent across variables
                ds = ds.unify_chunks()
                if 'latitude' in ds.dims and 'longitude' in ds.dims:
                    ds = ds.chunk({'latitude': 512, 'longitude': 512})

                #spatial & quality mask (lazy)
                mask = (
                    (ds['flag'] !=1) &
                    (ds['flag'] !=2) &
                    (ds['no2'] >0) &
                    (ds['longitude'] >= lon_min) &
                    (ds['longitude'] <= lon_max) &
                    (ds['latitude'] >= lat_min) &
                    (ds['latitude'] <= lat_max) )

                #mask 
                mask = mask.compute()
                filtered = ds.where(mask, drop=True)
                pdf =filtered.to_dataframe().reset_index()

                #keep all hours
                pdf['time'] = pd.to_datetime(pdf['time'], errors='coerce')
                pdf = pdf[pdf['time'].notnull()]
                pdf['date'] =pdf['time'].dt.date.astype(str)
                pdf['time'] =pdf['time'].dt.time.astype(str)

                if 'flag' in pdf.columns:
                    pdf = pdf.drop(columns=['flag'])

                #write parquet file per file
                output_file = os.path.join(temp_parquet_dir, f"{os.path.splitext(file)[0]}.parquet")
                pdf.to_parquet(output_file, index=False, engine='pyarrow')

        except Exception as e:
            print(f"there was an error processing this file {file}: {e}")

    #combine all per-file parquets into a single parquet "folder"
    print("making parquet folder with files")
    
    final_ddf = dd.read_parquet(temp_parquet_dir, engine='pyarrow', index=None)
    final_ddf.to_parquet(output_parquet, engine='pyarrow', write_index=False)
    print(f"all NO2 data has been saved to this folder: {output_parquet}")


In [6]:
folder_path = '/Volumes/share-sat2/SAT/TEMPO/NO2/L3/V03/2024/03'
output_parquet = "/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_03_folder" 

lon_min, lon_max = -104, -80 
lat_min, lat_max = 36, 49

process_no2_files_to_combined_parquet(folder_path, output_parquet, lon_min, lon_max, lat_min, lat_max)

Processing Files: 100%|███████████████████████████████████████| 436/436 [7:53:08<00:00, 65.11s/file]


making parquet folder with files
All NO2 data has been saved to /Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_03_folder


In [7]:
ddf = dd.read_parquet("/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_03_folder")

In [8]:
march_df = ddf.compute()

In [9]:
march_df.to_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_03_combined.parquet', engine='pyarrow', index=False)

In [3]:
folder_path = '/Volumes/share-sat2/SAT/TEMPO/NO2/L3/V03/2024/04'
output_parquet = "/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_04_folder" 

lon_min, lon_max = -104, -80 
lat_min, lat_max = 36, 49

process_no2_files_to_combined_parquet(folder_path, output_parquet, lon_min, lon_max, lat_min, lat_max)

Processing Files: 100%|█████████████████████████████████████████████████████| 458/458 [1:00:36<00:00,  7.94s/file]


making parquet folder with files
All NO2 data has been saved to /Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_04_folder


In [4]:
ddf = dd.read_parquet("/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_04_folder")

In [5]:
april_df = ddf.compute()

In [6]:
april_df.to_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_04_combined.parquet', engine='pyarrow', index=False)

In [7]:
folder_path = '/Volumes/share-sat2/SAT/TEMPO/NO2/L3/V03/2024/05'
output_parquet = "/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_05_folder" 

lon_min, lon_max = -104, -80 
lat_min, lat_max = 36, 49

process_no2_files_to_combined_parquet(folder_path, output_parquet, lon_min, lon_max, lat_min, lat_max)

Processing Files: 100%|█████████████████████████████████████████████████████| 534/534 [1:08:25<00:00,  7.69s/file]


making parquet folder with files
All NO2 data has been saved to /Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_05_folder


In [3]:
ddf = dd.read_parquet("/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_05_folder")

In [4]:
may_df = ddf.compute()

In [5]:
may_df.to_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_05_combined.parquet', engine='pyarrow', index=False)

In [3]:
folder_path = '/Volumes/share-sat2/SAT/TEMPO/NO2/L3/V03/2024/06'
output_parquet = "/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_06_folder" 

lon_min, lon_max = -104, -80 
lat_min, lat_max = 36, 49

process_no2_files_to_combined_parquet(folder_path, output_parquet, lon_min, lon_max, lat_min, lat_max)

Processing Files: 100%|█████████████████████████████████████████████████████| 523/523 [1:17:40<00:00,  8.91s/file]


making parquet folder with files
All NO2 data has been saved to /Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_06_folder


In [4]:
ddf = dd.read_parquet("/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_06_folder")

In [5]:
june_df = ddf.compute()

In [6]:
june_df.to_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_06_combined.parquet', engine='pyarrow', index=False)

In [3]:
folder_path = '/Volumes/share-sat2/SAT/TEMPO/NO2/L3/V03/2024/07'
output_parquet = "/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_07_folder" 

lon_min, lon_max = -104, -80 
lat_min, lat_max = 36, 49

process_no2_files_to_combined_parquet(folder_path, output_parquet, lon_min, lon_max, lat_min, lat_max)

Processing Files: 100%|█████████████████████████████████████████████████████| 517/517 [1:00:04<00:00,  6.97s/file]


making parquet folder with files
All NO2 data has been saved to /Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_07_folder


In [4]:
ddf = dd.read_parquet("/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_07_folder")

In [5]:
july_df = ddf.compute()

In [6]:
july_df.to_parquet('/Volumes/share-sat2/SAT/TEMPO/Toireasa/all_tempo_hours_no2_L3_VO3_2024_07_combined.parquet', engine='pyarrow', index=False)